# 00 — DLC SuperAnimal-Quadruped smoke test (sample horse video)

**Goal:** verify that DLC 3.x with the SuperAnimal-Quadruped model runs on a local Mac and lays keypoints visually sensibly on a sample horse clip.

**Output:** annotated MP4 in `../outputs/sample_with_keypoints.mp4` + 5 preview frames.

**Gate (see `../GATE.md`):** items 1 (setup) and 2 (sensible keypoints).

**Requirements:** `bash setup.sh` already run from the `poc/` folder.


## 1. Sanity check — versions, weights, sample video

In [ ]:
import os, sys, glob, platform
from pathlib import Path

POC = Path(os.getcwd()).resolve()
if POC.name == 'notebooks':
    POC = POC.parent
os.chdir(POC)
print('POC dir:', POC)
print('Python :', sys.version.split()[0])
print('Platform:', platform.platform())

import torch
print('Torch  :', torch.__version__, '| MPS dostepny:', torch.backends.mps.is_available())

import deeplabcut
print('DLC    :', deeplabcut.__version__)

sample = POC / 'data' / 'sample_horse.mp4'
weights_dir = POC / 'checkpoints' / 'superanimal-quadruped'
assert sample.exists(), f'BRAK sample video w {sample}. Uruchom bash setup.sh'
print('Sample :', sample, f'({sample.stat().st_size / 1e6:.1f} MB)')
print('Weights:', weights_dir, '(istnieje)' if weights_dir.exists() else '(BRAK — DLC pobierze lazy)')

## 2. SuperAnimal-Quadruped inference (zero-shot)

DLC 3.x ships `video_inference_superanimal`, which pulls the weights on first call (if missing), overlays keypoints, and writes both an annotated MP4 and a coordinate table (.h5).


In [ ]:
out_dir = POC / 'outputs'
out_dir.mkdir(exist_ok=True)

deeplabcut.video_inference_superanimal(
    videos=[str(sample)],
    superanimal_name='superanimal_quadruped',
    model_name='hrnet_w32',
    detector_name='fasterrcnn_resnet50_fpn_v2',
    video_adapt=False,
    dest_folder=str(out_dir),
    pseudo_threshold=0.1,
)

## 3. Visual inspection — 5 frames with overlay

In [ ]:
import cv2, matplotlib.pyplot as plt, numpy as np

annotated = sorted(out_dir.glob('*labeled.mp4'))
if not annotated:
    annotated = sorted(out_dir.glob('*_labeled*.mp4'))
assert annotated, f'Nie znaleziono annotated video w {out_dir}. Sprawdz logi powyzej.'
vid_path = str(annotated[0])
print('Annotated video:', vid_path)

cap = cv2.VideoCapture(vid_path)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f'Frames: {n_frames}, FPS: {fps:.1f}, duration: {n_frames/fps:.1f}s')

samples_idx = np.linspace(0, n_frames - 1, 5, dtype=int)
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, idx in zip(axes, samples_idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ok, frame = cap.read()
    if not ok:
        ax.set_title(f'klatka {idx} — brak')
        ax.axis('off')
        continue
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f'klatka {idx}')
    ax.axis('off')
cap.release()
plt.tight_layout()
plt.savefig(out_dir / 'sample_keypoints_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print('Zapisalem podglad do', out_dir / 'sample_keypoints_grid.png')

## 4. Decision note (entry into `../GATE.md`)

After reviewing `outputs/<sample>_labeled.mp4` in QuickTime / VLC and the `sample_keypoints_grid.png` grid, answer:

**Do the keypoints land visually sensibly on the horse (hooves, joints, eyes, ears) on ≥ 80% of frames?**

YES → GATE item 2 = YES. Move on to `01_read_my_ears_replicate.ipynb`.

NO → GATE item 2 = NO. Check that the sample has a horizontal camera and a single horse in frame. If keypoints still drift — that is a meaningful domain-shift signal; record in GATE.md and decide whether to try notebook 99 (Colab) or accept NO-GO.
